# Validation Tests

In [1]:
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "notebooks" / "00_common.ipynb").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook inside the Olist project folder.")
common_notebook = str(ROOT / "notebooks" / "00_common.ipynb")
%run "$common_notebook"

In [2]:
import tempfile
from decimal import Decimal

import pandas as pd
import pytest
from IPython.display import display

SYNTHETIC_RAW_DIR = ROOT / 'tests' / 'fixtures' / 'synthetic_olist'

In [3]:
def create_checks(tmp_path, failures=0):
    sample = ProjectPaths(tmp_path / "raw", tmp_path / "sample.duckdb", tmp_path / "outputs")
    with connect(sample) as con:
        con.execute("CREATE SCHEMA checks")
        con.execute("CREATE TABLE test_rows AS SELECT ?::BIGINT AS errors", [failures])
        con.execute("""
            CREATE VIEW checks.required_checks AS
            SELECT 'B' || LPAD(number::VARCHAR, 3, '0') AS check_id,
                   CASE WHEN number = 1 THEN errors ELSE 0 END AS affected_count
            FROM range(1, 12) AS numbers(number) CROSS JOIN test_rows
        """)
    return sample

In [4]:
def test_valid_data_passes_checks(tmp_path):
    sample = create_checks(tmp_path)
    check_data(sample)

In [5]:
def test_failed_checks_stop_export_before_writing_files(tmp_path):
    sample = create_checks(tmp_path, failures=2)
    with pytest.raises(RuntimeError, match="Data checks failed: B001"):
        export_tableau(sample, {"should_not_export": ("missing_table", "1")})
    assert not sample.output_dir.exists()

In [6]:
def test_missing_or_incomplete_checks_are_rejected(tmp_path):
    sample = create_checks(tmp_path)
    with connect(sample) as con:
        con.execute("DROP VIEW checks.required_checks")
    with pytest.raises(RuntimeError, match="Run data loading"):
        check_data(sample)
    with connect(sample) as con:
        con.execute("CREATE TABLE checks.required_checks AS SELECT 'B001' AS check_id, 0 AS affected_count")
    with pytest.raises(RuntimeError, match="missing or incomplete"):
        check_data(sample)

In [7]:
def test_checks_read_current_data(tmp_path):
    sample = create_checks(tmp_path)
    check_data(sample)
    with connect(sample) as con:
        con.execute("UPDATE test_rows SET errors = 1")
    with pytest.raises(RuntimeError, match="Data checks failed: B001"):
        check_data(sample)

In [8]:
def fetch_one(con, sql):
    result = con.execute(sql).fetchone()
    assert result is not None
    return result

In [9]:
def test_monthly_kpi_contract(synthetic_connection):
    result = fetch_one(
        synthetic_connection,
        """
        SELECT
            total_order_count,
            delivered_order_count,
            canceled_order_count,
            unavailable_order_count,
            delivery_sla_eligible_count,
            on_time_delivery_count,
            on_time_delivery_rate,
            handover_candidate_count,
            handover_sla_eligible_count,
            handover_breach_count,
            handover_breach_rate
        FROM marts.mart_monthly_fulfillment_kpi
        WHERE purchase_month = DATE '2018-01-01'
        """,
    )

    assert result[:6] == (8, 6, 1, 1, 6, 4)
    assert result[6] == pytest.approx(4 / 6)
    assert result[7:10] == (5, 4, 1)
    assert result[10] == pytest.approx(1 / 4)

In [10]:
def test_promised_calendar_day_is_on_time(synthetic_connection):
    on_time, date_variance, timestamp_would_be_late = fetch_one(
        synthetic_connection,
        """
        SELECT
            on_time_delivery,
            delivery_variance_days,
            order_delivered_customer_date > order_estimated_delivery_date
        FROM marts.mart_order_fulfillment
        WHERE order_id = 'ORDER_A'
        """,
    )

    assert on_time is True
    assert date_variance == 0
    assert timestamp_would_be_late is True

In [11]:
def test_late_delivery_and_handover_breach_are_separate_flags(synthetic_connection):
    on_time, handover_breach, delay_hours = fetch_one(
        synthetic_connection,
        """
        SELECT
            on_time_delivery,
            handover_sla_breach,
            maximum_positive_handover_delay_hours
        FROM marts.mart_order_seller
        WHERE order_id = 'ORDER_B' AND seller_id = 'SELLER_B'
        """,
    )

    assert on_time is False
    assert handover_breach is True
    assert delay_hours == pytest.approx(24.0)

In [12]:
def test_multi_seller_order_is_excluded_from_primary_seller_sla(synthetic_connection):
    rows = synthetic_connection.execute(
        """
        SELECT
            seller_id,
            seller_attribution_reliable,
            is_handover_sla_observable,
            is_handover_sla_eligible
        FROM marts.mart_order_seller
        WHERE order_id = 'ORDER_C'
        ORDER BY seller_id
        """,
    ).fetchall()

    assert rows == [
        ("SELLER_A", False, True, False),
        ("SELLER_B", False, True, False),
    ]

In [13]:
def test_missing_shipping_deadline_reduces_handover_denominator(synthetic_connection):
    candidate, eligible, deadline_count = fetch_one(
        synthetic_connection,
        """
        SELECT
            is_handover_candidate,
            is_handover_sla_eligible,
            shipping_limit_nonnull_count
        FROM marts.mart_order_seller
        WHERE order_id = 'ORDER_D'
        """,
    )

    assert candidate is True
    assert eligible is False
    assert deadline_count == 0

In [14]:
def test_invalid_precarrier_sequence_only_excludes_affected_duration(synthetic_connection):
    precarrier_valid, delivery_eligible, on_time = fetch_one(
        synthetic_connection,
        """
        SELECT
            is_precarrier_duration_valid,
            is_delivery_sla_eligible,
            on_time_delivery
        FROM marts.mart_order_fulfillment
        WHERE order_id = 'ORDER_G'
        """,
    )

    assert precarrier_valid is False
    assert delivery_eligible is True
    assert on_time is True

In [15]:
def test_one_to_many_sources_do_not_inflate_order_values(synthetic_connection):
    result = fetch_one(
        synthetic_connection,
        """
        SELECT
            item_count,
            gross_item_value,
            freight_value,
            payment_row_count,
            payment_total,
            review_row_count
        FROM marts.mart_order_fulfillment
        WHERE order_id = 'ORDER_H'
        """,
    )

    assert result == (
        2,
        Decimal("120.00"),
        Decimal("12.00"),
        2,
        Decimal("132.00"),
        2,
    )

    source_total, mart_total = fetch_one(
        synthetic_connection,
        """
        SELECT
            (SELECT SUM(price) FROM staging.stg_order_items),
            (SELECT SUM(gross_item_value) FROM marts.mart_order_fulfillment)
        """,
    )
    assert source_total == Decimal("540.00")
    assert mart_total == source_total

In [16]:
def test_latest_answered_review_is_representative(synthetic_connection):
    review_count, representative_id, representative_score = fetch_one(
        synthetic_connection,
        """
        SELECT
            review_row_count,
            representative_review_id,
            representative_review_score
        FROM marts.mart_order_fulfillment
        WHERE order_id = 'ORDER_H'
        """,
    )

    assert review_count == 2
    assert representative_id == "REVIEW_NEW"
    assert representative_score == 2

In [17]:
def test_handover_population_sensitivity_contract(synthetic_connection):
    rows = synthetic_connection.execute(
        """
        SELECT population_name, candidate_count, eligible_count, breach_count
        FROM analysis.handover_population_sensitivity
        ORDER BY population_name
        """,
    ).fetchall()

    assert rows == [
        ("DELIVERED_ONLY_SINGLE_SELLER", 5, 4, 1),
        ("MULTI_SELLER_OBSERVABLE", 2, 2, 0),
        ("PRIMARY_SINGLE_SELLER", 5, 4, 1),
    ]

In [18]:
def test_platform_context_baseline_uses_same_eligible_population(synthetic_connection):
    platform_late_rate, platform_low_review_rate = fetch_one(
        synthetic_connection,
        """
        SELECT MAX(platform_late_delivery_rate), MAX(platform_low_review_rate)
        FROM analysis.agg_seller_performance
        """,
    )
    expected_late_rate, expected_low_review_rate = fetch_one(
        synthetic_connection,
        """
        SELECT
            COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
                / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0),
            COUNT(*) FILTER (
                WHERE has_valid_review AND representative_review_score <= 2
            )::DOUBLE / NULLIF(COUNT(*) FILTER (WHERE has_valid_review), 0)
        FROM marts.mart_order_seller
        WHERE is_handover_sla_eligible
        """,
    )

    assert platform_late_rate == pytest.approx(expected_late_rate)
    assert platform_low_review_rate == pytest.approx(expected_low_review_rate)

In [19]:
def test_category_delivery_performance_contract(synthetic_connection):
    rows = synthetic_connection.execute(
        """
        SELECT category_name, category_involved_order_count,
               delivery_sla_eligible_count, late_delivery_order_count,
               late_delivery_rate
        FROM analysis.agg_category_performance
        WHERE category_name IN ('category_a_en', 'category_b_en')
        ORDER BY category_name
        """,
    ).fetchall()

    assert rows[0][:4] == ("category_a_en", 5, 4, 1)
    assert rows[0][4] == pytest.approx(1 / 4)
    assert rows[1][:4] == ("category_b_en", 3, 3, 2)
    assert rows[1][4] == pytest.approx(2 / 3)

In [20]:
def test_analysis_grains_are_unique(synthetic_connection):
    seller_region_counts = fetch_one(
        synthetic_connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (seller_id, customer_state))
        FROM analysis.agg_seller_customer_state_lane
        """,
    )
    stage_outcome_counts = fetch_one(
        synthetic_connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (delivery_outcome, stage_name))
        FROM analysis.fulfillment_stage_by_delivery_outcome
        """,
    )
    category_counts = fetch_one(
        synthetic_connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT category_name)
        FROM analysis.agg_category_performance
        """,
    )
    category_stage_counts = fetch_one(
        synthetic_connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (category_name, stage_name))
        FROM analysis.category_fulfillment_stage_performance
        """,
    )

    assert seller_region_counts[0] == seller_region_counts[1]
    assert stage_outcome_counts == (8, 8)
    assert category_counts[0] == category_counts[1]
    assert category_stage_counts[0] == category_stage_counts[1]
    assert category_stage_counts[0] == category_counts[0] * 4

In [21]:
def scalar(con, sql):
    return fetch_one(con, sql)[0]

In [22]:
def test_source_and_primary_order_counts(connection):
    assert scalar(connection, "SELECT COUNT(*) FROM staging.stg_orders") == 99_441
    assert scalar(connection, "SELECT COUNT(*) FROM marts.mart_order_fulfillment") == 99_441
    assert (
        scalar(
            connection,
            """
            SELECT COUNT(*)
            FROM marts.mart_order_fulfillment
            WHERE is_primary_analysis_period
            """,
        )
        == 99_092
    )

In [23]:
def test_mart_grains_are_unique(connection):
    order_counts = fetch_one(
        connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT order_id)
        FROM marts.mart_order_fulfillment
        """,
    )
    seller_counts = fetch_one(
        connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (order_id, seller_id))
        FROM marts.mart_order_seller
        """,
    )
    category_counts = fetch_one(
        connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (order_id, category_name))
        FROM marts.mart_order_category
        """,
    )
    assert order_counts == (99_441, 99_441)
    assert seller_counts == (100_010, 100_010)
    assert category_counts[0] == category_counts[1]

In [24]:
def test_category_analysis_grains_and_value_control(connection):
    category_counts = fetch_one(
        connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT category_name)
        FROM analysis.agg_category_performance
        """,
    )
    category_stage_counts = fetch_one(
        connection,
        """
        SELECT COUNT(*), COUNT(DISTINCT (category_name, stage_name))
        FROM analysis.category_fulfillment_stage_performance
        """,
    )
    category_value, order_value = fetch_one(
        connection,
        """
        SELECT
            (SELECT SUM(category_gross_item_value) FROM marts.mart_order_category),
            (SELECT SUM(gross_item_value) FROM marts.mart_order_fulfillment)
        """,
    )

    assert category_counts[0] == category_counts[1]
    assert category_stage_counts[0] == category_stage_counts[1]
    assert category_stage_counts[0] == category_counts[0] * 4
    assert category_value == order_value

In [25]:
def test_calendar_date_delivery_rule_preserves_promised_day(connection):
    promised_day_deliveries = scalar(
        connection,
        """
        SELECT COUNT(*)
        FROM marts.mart_order_fulfillment
        WHERE is_delivery_sla_eligible
          AND order_delivered_customer_date::DATE = order_estimated_delivery_date::DATE
          AND order_delivered_customer_date > order_estimated_delivery_date
          AND on_time_delivery
        """,
    )
    assert promised_day_deliveries == 1_292

In [26]:
def test_primary_handover_population_is_single_seller(connection):
    unreliable_eligible = scalar(
        connection,
        """
        SELECT COUNT(*)
        FROM marts.mart_order_seller
        WHERE is_handover_sla_eligible AND NOT seller_attribution_reliable
        """,
    )
    assert unreliable_eligible == 0

In [27]:
def test_required_data_checks_pass(connection):
    checks, affected = fetch_one(
        connection,
        """
        SELECT COUNT(*), SUM(affected_count) FROM checks.required_checks
        """,
    )
    assert checks == 11
    assert affected == 0

In [28]:
def test_known_data_issue_counts(connection):
    actual = dict(
        connection.execute(
            """
            SELECT check_id, affected_count
            FROM checks.data_notes
            """
        ).fetchall()
    )
    assert actual == {
        "W001": 1_350,
        "W002": 23,
        "W003": 8,
        "W004": 1_278,
        "W005": 727,
        "W006": 547,
        "W007": 278,
        "W008": 0,
        "W009": 1_292,
        "W010": 220,
        "W011": 0,
        "W012": 1_603,
        "W013": 303,
        "W014": 20,
        "W015": 767,
    }

In [29]:
def test_missing_source_stops_load(tmp_path):
    sample = ProjectPaths(tmp_path / "raw", tmp_path / "missing.duckdb", tmp_path / "outputs")
    with pytest.raises(FileNotFoundError, match="Missing source files"):
        run_phase(sample, "load")
    assert not sample.db_path.exists()

In [30]:
def test_failed_phase_rolls_back(tmp_path):
    sample = ProjectPaths(SYNTHETIC_RAW_DIR, tmp_path / "rollback.duckdb", tmp_path / "outputs")
    with connect(sample) as con:
        con.execute("CREATE TABLE rollback_probe AS SELECT 1 AS value")
    with pytest.MonkeyPatch.context() as patch:
        patch.setitem(SQL_SCRIPTS, "10_intermediate", "UPDATE rollback_probe SET value = 2;")
        patch.setitem(SQL_SCRIPTS, "20_marts", "SELECT * FROM missing_table_for_rollback_test;")
        with pytest.raises(duckdb.Error):
            run_phase(sample, "build")
    with connect(sample, read_only=True) as con:
        assert fetch_one(con, "SELECT value FROM rollback_probe") == (1,)

In [31]:
passed = []
with tempfile.TemporaryDirectory() as temporary:
    temp = Path(temporary).resolve()
    synthetic_paths = ProjectPaths(SYNTHETIC_RAW_DIR, temp / "synthetic.duckdb", temp / "outputs")
    for phase in ("load", "build", "analyze"):
        run_phase(synthetic_paths, phase)
    with connect(synthetic_paths, read_only=True) as synthetic_connection:
        for test in [
            test_monthly_kpi_contract,
            test_promised_calendar_day_is_on_time,
            test_late_delivery_and_handover_breach_are_separate_flags,
            test_multi_seller_order_is_excluded_from_primary_seller_sla,
            test_missing_shipping_deadline_reduces_handover_denominator,
            test_invalid_precarrier_sequence_only_excludes_affected_duration,
            test_one_to_many_sources_do_not_inflate_order_values,
            test_latest_answered_review_is_representative,
            test_handover_population_sensitivity_contract,
            test_platform_context_baseline_uses_same_eligible_population,
            test_category_delivery_performance_contract,
            test_analysis_grains_are_unique,
        ]:
            test(synthetic_connection)
            passed.append(test.__name__)
    for test in [
        test_valid_data_passes_checks,
        test_failed_checks_stop_export_before_writing_files,
        test_missing_or_incomplete_checks_are_rejected,
        test_checks_read_current_data,
        test_missing_source_stops_load,
        test_failed_phase_rolls_back,
    ]:
        tmp_path = temp / test.__name__
        tmp_path.mkdir()
        test(tmp_path)
        passed.append(test.__name__)

with connect(paths, read_only=True) as connection:
    for test in [
        test_source_and_primary_order_counts,
        test_mart_grains_are_unique,
        test_category_analysis_grains_and_value_control,
        test_calendar_date_delivery_rule_preserves_promised_day,
        test_primary_handover_population_is_single_seller,
        test_required_data_checks_pass,
        test_known_data_issue_counts,
    ]:
        test(connection)
        passed.append(test.__name__)

print(f"{len(passed)} tests passed")
display(pd.DataFrame({"test": passed, "status": "PASS"}))

[load] 00_staging
[build] 10_intermediate
[build] 20_marts
[analyze] 40_analysis


[build] 10_intermediate
25 tests passed


,test,status
0,test_monthly_kpi_contract,PASS
1,test_promised_calendar_day_is_on_time,PASS
2,test_late_delivery_and_handover_breach_are_sep...,PASS
3,test_multi_seller_order_is_excluded_from_prima...,PASS
4,test_missing_shipping_deadline_reduces_handove...,PASS
5,test_invalid_precarrier_sequence_only_excludes...,PASS
6,test_one_to_many_sources_do_not_inflate_order_...,PASS
7,test_latest_answered_review_is_representative,PASS
8,test_handover_population_sensitivity_contract,PASS
9,test_platform_context_baseline_uses_same_eligi...,PASS
